# RelCheck — Enriched Pipeline (100 Images, BLIP-2)

## Architecture
1. **BLIP-2** generates captions (weak captioner → many errors to fix)
2. **GroundingDINO** detects objects + bboxes (deterministic)
3. **Maverick VLM** describes relationships between detected objects
4. **Llama-3.3-70B** analyzes caption vs KB → finds errors + missing facts
5. **Llama-3.3-70B** rewrites caption: fixes errors, adds missing facts, ≤3 sentences
6. **Llama-3.3-70B** verifies rewrite is faithful to KB
7. **R-POPE (LLM-judge)** evaluates: does enriched caption answer R-Bench questions better?

## Key Insight
BLIP-2 captions are short and vague (53% R-POPE accuracy).
Most errors are OMISSIONS (caption doesn't mention objects/relations the KB detects).
Enrichment addresses both false claims AND omissions using verified KB evidence.

## Expected Runtime
- Cell 0-1: ~3 min (model loading)
- Cell 2: ~1 min (image loading)
- Cell 3: ~2 min (BLIP-2 captioning, local GPU)
- Cell 4: ~15 min (KB: GroundingDINO local + 100 Maverick API calls)
- Cell 5: ~15 min (analysis + rewrite: 200 Llama API calls)
- Cell 6: ~20 min (R-POPE: ~550 Llama API calls)
- **Total: ~60 min**


In [ ]:
# ============================================================
# Cell 0 — Install + Setup
# ============================================================
!pip install -q together transformers pillow torch torchvision
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, random
from io import BytesIO
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import torch
from together import Together
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    Blip2Processor, Blip2ForConditionalGeneration,
)

# ── API Key ──
TOGETHER_API_KEY = ""  # <-- paste your key here
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

# ── Model IDs ──
VLM_MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"  # VLM for KB
LLM_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"            # Compare + Correct
GDINO_ID  = "IDEA-Research/grounding-dino-tiny"                    # Object detection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device:    {DEVICE}")
print(f"Verifier:  {VLM_MODEL}")
print(f"LLM:       {LLM_MODEL}")
print(f"GDino:     {GDINO_ID}")


In [ ]:
# ============================================================
# Cell 1 — Load Models (GroundingDINO + BLIP-2)
# ============================================================
print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print("  GroundingDINO loaded.")

print("Loading BLIP-2...")
blip2_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl", torch_dtype=torch.float16, device_map="auto"
)
print("  BLIP-2 loaded.")
print(f"\nReady on {DEVICE}.")


In [ ]:
# ============================================================
# Cell 2 — Load Images + R-Bench Data
# ============================================================
from google.colab import drive
from pathlib import Path

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"

# Load R-Bench
with open(RBENCH_PATH) as f:
    rbench_data = json.load(f)
print(f"R-Bench: {len(rbench_data)} questions total")

# Build lookup by image
rbench_by_image = defaultdict(list)
for entry in rbench_data:
    rbench_by_image[entry['image']].append(entry)
print(f"R-Bench covers {len(rbench_by_image)} unique images")

# Find images on Drive with R-Bench questions
cached_images = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + \
                list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
cached_stems = {p.stem: p for p in cached_images}

matched_images = {}
for rb_key in rbench_by_image:
    rb_clean = rb_key.replace('.jpg', '').replace('.jpeg', '').replace('.png', '')
    if rb_clean in cached_stems:
        matched_images[rb_clean] = {
            "path": str(cached_stems[rb_clean]),
            "questions": rbench_by_image[rb_key],
        }

print(f"Images on Drive with R-Bench questions: {len(matched_images)}")

# Sample N images
N_IMAGES = 100
random.seed(42)
selected_ids = random.sample(list(matched_images.keys()), min(N_IMAGES, len(matched_images)))

# Load images into memory + keep question mapping
images = {}
img_to_questions = {}
for img_id in selected_ids:
    info = matched_images[img_id]
    images[img_id] = Image.open(info["path"]).convert("RGB")
    img_to_questions[img_id] = info["questions"]

total_q = sum(len(qs) for qs in img_to_questions.values())
print(f"\nLoaded {len(images)} images with {total_q} R-Bench questions")
print(f"Avg {total_q/len(images):.1f} questions per image")


In [ ]:
# ============================================================
# Cell 3 — Stage 1: Caption Generation (BLIP-2)
# ============================================================

BLIP2_PROMPT = "Describe the relationships between objects and people in this image."

def caption_with_blip2(image):
    """Generate caption via BLIP-2 on Colab GPU."""
    inputs = blip2_processor(images=image, text=BLIP2_PROMPT, return_tensors="pt").to(
        blip2_model.device, torch.float16
    )
    with torch.no_grad():
        out = blip2_model.generate(**inputs, max_new_tokens=80, temperature=0.7)
    return blip2_processor.decode(out[0], skip_special_tokens=True).strip()

captions = {}
for i, (img_id, img) in enumerate(images.items()):
    try:
        cap = caption_with_blip2(img)
        captions[img_id] = cap
    except Exception as e:
        captions[img_id] = ""
        print(f"  [{img_id[:15]}] FAILED: {e}")

    if (i + 1) % 20 == 0 or i == 0:
        print(f"[{i+1}/{len(images)}] {img_id[:15]}: {captions[img_id][:80]}...")

print(f"\nCaptioned {sum(1 for c in captions.values() if c)}/{len(images)} images")


In [ ]:
# ============================================================
# Cell 4 — Stage 2: Visual Knowledge Base Construction
# ============================================================
# Three layers of evidence per image:
#   HARD  — GroundingDINO: objects, counts, bboxes (deterministic)
#   GEOM  — Bbox geometry: spatial relationships (deterministic)
#   SOFT  — Maverick VLM: actions, attributes, relationships (visual)
#
# This runs on 100 images. GroundingDINO is fast (~1s/image).
# Maverick VLM via API is the bottleneck (~2s/image).
# Expected: ~5 min total.

import spacy
nlp = spacy.load("en_core_web_sm")

DETECTION_THRESHOLD = 0.25

BROAD_CATEGORIES = [
    "person", "man", "woman", "child", "boy", "girl",
    "dog", "cat", "bird", "horse", "animal",
    "car", "bicycle", "motorcycle", "bus", "truck",
    "chair", "table", "bench", "couch", "bed",
    "food", "plate", "bowl", "cup", "bottle", "glass",
    "bag", "umbrella", "phone", "book", "sign",
    "hat", "jacket", "vest", "helmet", "glasses",
    "tree", "flower", "grass", "water",
]


def _clean_gdino_label(label):
    """Clean noisy GroundingDINO labels like 'bowl a bowl' or 'a cup cup'."""
    label = label.strip().lower()
    words = label.split()
    if not words:
        return label
    # Strip leading articles
    while words and words[0] in ('a', 'an', 'the'):
        words = words[1:]
    if not words:
        return label
    # Remove duplicate words
    seen = []
    for w in words:
        if w not in seen:
            seen.append(w)
    return ' '.join(seen) if seen else label


def detect_objects(image, text_queries, threshold=DETECTION_THRESHOLD):
    """Run GroundingDINO. Returns list of (label, score, bbox_norm)."""
    text = ". ".join(text_queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=threshold, text_threshold=threshold,
        target_sizes=[image.size[::-1]],
    )[0]
    detections = []
    W, H = image.size
    lkey = "text_labels" if "text_labels" in results else "labels"
    for score, label, box in zip(results["scores"], results[lkey], results["boxes"]):
        x1, y1, x2, y2 = box.tolist()
        label = _clean_gdino_label(label)
        detections.append((label, score.item(), [x1/W, y1/H, x2/W, y2/H]))
    return detections


def extract_nouns_from_caption(caption):
    """Extract noun phrases from caption for targeted detection."""
    doc = nlp(caption)
    nouns = set()
    for chunk in doc.noun_chunks:
        nouns.add(chunk.root.text.lower())
        nouns.add(chunk.text.lower().strip())
    return list(nouns)


def compute_spatial_facts(detections):
    """Compute pairwise spatial relationships from bboxes."""
    facts = []
    counts = Counter(label for label, _, _ in detections)

    for label, count in counts.items():
        facts.append(f"There {'are' if count > 1 else 'is'} {count} '{label}'")

    unique_labels = list(counts.keys())
    for i, l1 in enumerate(unique_labels):
        det1 = [(s, b) for l, s, b in detections if l == l1]
        for j, l2 in enumerate(unique_labels):
            if i >= j:
                continue
            det2 = [(s, b) for l, s, b in detections if l == l2]
            _, b1 = max(det1, key=lambda x: x[0])
            _, b2 = max(det2, key=lambda x: x[0])
            c1x, c1y = (b1[0]+b1[2])/2, (b1[1]+b1[3])/2
            c2x, c2y = (b2[0]+b2[2])/2, (b2[1]+b2[3])/2
            # Containment
            contained = (b1[0] >= b2[0]-0.05 and b1[2] <= b2[2]+0.05 and
                         b1[1] >= b2[1]-0.05 and b1[3] <= b2[3]+0.05)
            if contained:
                facts.append(f"'{l1}' is on/inside '{l2}' (spatially contained)")
                continue
            dx, dy = c1x - c2x, c1y - c2y
            if abs(dx) > abs(dy):
                pos = "to the left of" if dx < 0 else "to the right of"
            else:
                pos = "above" if dy < 0 else "below"
            facts.append(f"'{l1}' is {pos} '{l2}'")
    return facts


def encode_image_b64(image):
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


MAVERICK_KB_PROMPT = """Describe the RELATIONSHIPS between objects in this image.

An object detector found these objects:
{detection_list}

For each pair of detected objects that interact, describe their relationship:
- SPATIAL: Where are they relative to each other? (on, next to, behind, inside, etc.)
- ACTIONS: What is each person/animal doing? What are they interacting with?
- ATTRIBUTES: Relevant attributes tied to relationships (wearing X, holding Y)

RULES:
- Only describe what you can clearly see
- Focus on relationships between the DETECTED objects listed above
- Use short, factual statements
- Do NOT list objects that aren't interacting

Format as a numbered list of relationship facts."""


def dedup_detections(detections):
    """Remove overlapping detections (IoU > 0.5), keep highest confidence."""
    seen = {}
    for label, score, bbox in sorted(detections, key=lambda x: -x[1]):
        key = label.lower().strip()
        if key not in seen:
            seen[key] = [(score, bbox)]
        else:
            is_dup = False
            for _, eb in seen[key]:
                ix1 = max(bbox[0], eb[0])
                iy1 = max(bbox[1], eb[1])
                ix2 = min(bbox[2], eb[2])
                iy2 = min(bbox[3], eb[3])
                inter = max(0, ix2-ix1) * max(0, iy2-iy1)
                a1 = (bbox[2]-bbox[0])*(bbox[3]-bbox[1])
                a2 = (eb[2]-eb[0])*(eb[3]-eb[1])
                iou = inter / (a1 + a2 - inter + 1e-8)
                if iou > 0.5:
                    is_dup = True
                    break
            if not is_dup:
                seen[key].append((score, bbox))
    clean = []
    for label, entries in seen.items():
        for score, bbox in entries:
            clean.append((label, score, bbox))
    return clean


# ── Build KB for all images ──
knowledge_bases = {}

for idx, (img_id, img) in enumerate(images.items()):
    kb = {"hard_facts": [], "spatial_facts": [], "visual_description": "", "detections": []}

    # Extract nouns from caption + broad categories
    caption_nouns = extract_nouns_from_caption(captions.get(img_id, ""))
    all_queries = list(set(caption_nouns + BROAD_CATEGORIES))

    # GroundingDINO detection
    raw_detections = detect_objects(img, all_queries)
    clean_detections = dedup_detections(raw_detections)
    kb["detections"] = clean_detections

    det_counts = Counter(l for l, _, _ in clean_detections)
    det_list_str = ""
    for label, count in det_counts.most_common():
        bboxes = [(s, b) for l, s, b in clean_detections if l == label]
        bbox_strs = [f"conf={s:.2f}" for s, b in bboxes]
        det_list_str += f"- {label} ({count}x): {', '.join(bbox_strs)}\n"

    kb["hard_facts"].append(f"Detected {len(clean_detections)} objects total:")
    for label, count in det_counts.most_common():
        kb["hard_facts"].append(f"  {count}x {label}")

    # Spatial geometry
    kb["spatial_facts"] = compute_spatial_facts(clean_detections)

    # Maverick VLM description
    b64 = encode_image_b64(img)
    try:
        resp = client.chat.completions.create(
            model=VLM_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": MAVERICK_KB_PROMPT.format(detection_list=det_list_str)},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ]}],
            temperature=0.0, max_tokens=500,
        )
        kb["visual_description"] = resp.choices[0].message.content.strip()
    except Exception as e:
        kb["visual_description"] = ""
        print(f"  [{img_id[:15]}] Maverick failed: {e}")

    knowledge_bases[img_id] = kb

    if (idx + 1) % 20 == 0 or idx == 0:
        print(f"[{idx+1}/{len(images)}] {img_id[:15]}: {len(clean_detections)} objects, "
              f"{len(kb['spatial_facts'])} spatial facts")
    time.sleep(0.3)

print(f"\nBuilt knowledge bases for {len(knowledge_bases)} images.")


In [ ]:
# ============================================================
# Cell 5 — Stage 3: Caption Analysis + Enriched Rewrite
# ============================================================
# Combined analysis + rewrite in one cell to reduce API calls.
# For each image:
#   1. Llama compares caption vs KB → finds contradictions + missing facts
#   2. Llama rewrites caption: fix errors + add missing facts, ≤3 sentences
#   3. Verification: check rewrite didn't invent unsupported claims

ANALYSIS_AND_REWRITE_PROMPT = """You are a caption quality improver. You have an image caption and a Visual Knowledge Base (KB) built from the actual image.

CAPTION: "{caption}"

=== VISUAL KNOWLEDGE BASE ===

DETECTED OBJECTS (from object detector — highly reliable):
{hard_facts}

SPATIAL RELATIONSHIPS (from bounding box geometry — deterministic):
{spatial_facts}

VISUAL DESCRIPTION (from vision model looking at the image):
{visual_description}

=== TASK ===

Step 1: Identify problems with the caption:
  - ERRORS: Claims in the caption that the KB contradicts
  - MISSING: Important facts in the KB that the caption doesn't mention (objects, relationships, attributes)

Step 2: Write an IMPROVED caption that:
  - Fixes any errors using KB evidence
  - Adds the most important missing details from the KB
  - Keeps everything the original caption got right
  - Is exactly 2-3 sentences long
  - Sounds natural (not a list of objects)
  - Only includes facts supported by the KB — do NOT invent anything

Output valid JSON:
{{
  "errors": [
    {{"claim": "what caption says", "correction": "what KB shows instead"}}
  ],
  "missing": [
    {{"fact": "important KB fact not in caption", "source": "which KB layer"}}
  ],
  "improved_caption": "The rewritten 2-3 sentence caption."
}}"""


VERIFY_PROMPT = """Check if this rewritten caption is faithful to the KB evidence.

Original: "{original}"
Rewritten: "{rewritten}"
KB objects detected: {objects}
KB relationships: {relationships}

Check:
1. Does the rewrite only contain facts supported by the KB?
2. Did it preserve correct details from the original?
3. Is it 2-3 sentences?

Answer ONLY "PASS" or "FAIL: [reason]"."""


def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2)+1)
    for c1 in s1:
        curr = [prev[0]+1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[-1]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]


def format_kb_text(kb):
    hard = "\n".join(f"- {f}" for f in kb["hard_facts"])
    spatial = "\n".join(f"- {f}" for f in kb["spatial_facts"]) if kb["spatial_facts"] else "- No spatial facts"
    visual = kb["visual_description"] if kb["visual_description"] else "- No visual description available"
    return hard, spatial, visual


final_results = {}

for idx, (img_id, img) in enumerate(images.items()):
    caption = captions.get(img_id, "")
    if not caption:
        final_results[img_id] = {
            "caption": "", "corrected": "",
            "errors": [], "missing": [],
            "edit_rate": 0.0, "status": "skipped_empty",
        }
        continue

    kb = knowledge_bases[img_id]
    hard, spatial, visual = format_kb_text(kb)

    # ── Combined analysis + rewrite ──
    prompt = ANALYSIS_AND_REWRITE_PROMPT.format(
        caption=caption, hard_facts=hard,
        spatial_facts=spatial, visual_description=visual,
    )

    improved = caption  # Default: keep original
    errors_found = []
    missing_found = []

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0, max_tokens=1000,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'```\s*$', '', raw)
        # Fix truncated JSON
        if raw.count('{') > raw.count('}'):
            raw += '"}'
        result = json.loads(raw)

        errors_found = result.get("errors", [])
        missing_found = result.get("missing", [])
        improved = result.get("improved_caption", caption)

        # Clean up improved caption
        improved = improved.strip().strip('"').strip("'")
        if not improved or len(improved) < 10:
            improved = caption

    except Exception as e:
        print(f"  [{img_id[:15]}] Analysis failed: {e}")

    # ── Safety checks ──
    edit_rate = levenshtein(caption, improved) / max(len(caption), 1)

    if edit_rate > 0.80:
        print(f"  [{img_id[:15]}] Edit rate {edit_rate:.0%} > 80% — too aggressive, keeping original")
        improved = caption
        edit_rate = 0.0
    elif improved != caption and (errors_found or missing_found):
        # ── Verification ──
        det_counts = Counter(l for l, _, _ in kb.get("detections", []))
        objects_str = ", ".join(f"{c}x {l}" for l, c in det_counts.most_common(10))
        rels_str = kb["visual_description"][:300] if kb["visual_description"] else "none"

        try:
            verify_resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": VERIFY_PROMPT.format(
                    original=caption, rewritten=improved,
                    objects=objects_str, relationships=rels_str,
                )}],
                temperature=0.0, max_tokens=50,
            )
            verdict = verify_resp.choices[0].message.content.strip()
            if verdict.upper().startswith("FAIL"):
                print(f"  [{img_id[:15]}] VERIFY FAILED: {verdict}")
                improved = caption
                edit_rate = 0.0
        except Exception as e:
            pass  # Keep rewrite on verification error

    final_results[img_id] = {
        "caption": caption,
        "corrected": improved,
        "errors": errors_found,
        "missing": missing_found,
        "edit_rate": edit_rate,
        "status": "modified" if improved != caption else "unchanged",
    }

    if (idx + 1) % 20 == 0 or idx == 0:
        n_err = len(errors_found)
        n_miss = len(missing_found)
        changed = "CHANGED" if improved != caption else "same"
        print(f"[{idx+1}/{len(images)}] {img_id[:15]}: {n_err} errors, {n_miss} missing → {changed} (edit {edit_rate:.0%})")

    time.sleep(0.3)

# ── Summary ──
n_modified = sum(1 for r in final_results.values() if r["status"] == "modified")
n_errors = sum(len(r["errors"]) for r in final_results.values())
n_missing = sum(len(r["missing"]) for r in final_results.values())
avg_edit = np.mean([r["edit_rate"] for r in final_results.values() if r["status"] == "modified"]) if n_modified > 0 else 0

print(f"\n{'='*60}")
print(f"ENRICHMENT SUMMARY")
print(f"{'='*60}")
print(f"  Images modified:  {n_modified}/{len(final_results)}")
print(f"  Total errors found: {n_errors}")
print(f"  Total missing facts: {n_missing}")
print(f"  Avg edit rate (modified only): {avg_edit:.0%}")


In [ ]:
# ============================================================
# Cell 6 — R-POPE Evaluation (LLM-Judge)
# ============================================================
# For each image with R-Bench questions:
#   1. Llama answers based on ORIGINAL caption → compare to GT
#   2. Llama answers based on ENRICHED caption → compare to GT
#   3. Track: improved, regressed, unchanged per question

RPOPE_PROMPT = """Based ONLY on this caption, answer the question with 'yes' or 'no'.
Do not use any external knowledge. Only use information stated in the caption.

Caption: "{caption}"
Question: {question}

Answer (yes or no):"""

def rpope_judge(caption, question):
    """Ask Llama to answer a question based solely on the caption."""
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content":
                RPOPE_PROMPT.format(caption=caption, question=question)}],
            temperature=0.0, max_tokens=5,
        )
        answer = resp.choices[0].message.content.strip().lower()
        if "yes" in answer and "no" not in answer:
            return "yes"
        elif "no" in answer:
            return "no"
        return "unknown"
    except Exception as e:
        return "unknown"


# ── Classify question type ──
def classify_question(question):
    q = question.lower()
    action_verbs = [
        "playing", "holding", "eating", "sitting", "standing", "running",
        "walking", "riding", "driving", "reading", "closing", "opening",
        "looking", "talking", "carrying", "throwing", "catching", "cooking",
        "swimming", "flying", "climbing", "jumping", "dancing", "sleeping",
        "drinking", "writing", "pointing", "pulling", "pushing", "cutting",
        "hitting", "kicking", "hugging", "kissing", "fighting", "leaning",
        "lying", "bending", "reaching", "touching", "feeding", "chasing",
        "surfing", "skiing", "skating", "smiling", "crying", "waving",
    ]
    for verb in action_verbs:
        if verb in q:
            return "ACTION"
    spatial_phrases = [
        " on a ", " on the ", " on top ", " in a ", " in the ",
        " inside ", " outside ", " near ", " next to ", " behind ",
        " above ", " below ", " under ", " beside ", " between ",
        " in front of ", " across ", " along ", " around ",
    ]
    for phrase in spatial_phrases:
        if phrase in f" {q} ":
            return "SPATIAL"
    attribute_kw = [
        "wearing", "color", "colour", "red", "blue", "green", "yellow",
        "black", "white", "brown", "large", "small", "big", "tall", "short",
        "old", "young", "made of", "type of", "kind of",
    ]
    for kw in attribute_kw:
        if kw in q:
            return "ATTRIBUTE"
    return "OTHER"


# ── Run R-POPE ──
orig_correct, enr_correct, total_q = 0, 0, 0
improved_questions = []
regressed_questions = []
unchanged_questions = []
per_image_results = []

# Track by question type
type_stats = defaultdict(lambda: {"orig_correct": 0, "enr_correct": 0, "total": 0})

for idx, img_id in enumerate(images):
    if img_id not in img_to_questions:
        continue

    questions = img_to_questions[img_id]
    r = final_results[img_id]
    orig_cap = r["caption"]
    enr_cap = r["corrected"]
    was_modified = r["status"] == "modified"

    img_orig_c, img_enr_c, img_total = 0, 0, 0

    for entry in questions[:10]:
        question = entry["question"]
        gt = entry["answer"]
        qtype = classify_question(question)

        orig_ans = rpope_judge(orig_cap, question)
        enr_ans = rpope_judge(enr_cap, question)

        orig_ok = (orig_ans == gt)
        enr_ok = (enr_ans == gt)

        if orig_ok: orig_correct += 1; img_orig_c += 1
        if enr_ok: enr_correct += 1; img_enr_c += 1
        total_q += 1; img_total += 1

        type_stats[qtype]["total"] += 1
        if orig_ok: type_stats[qtype]["orig_correct"] += 1
        if enr_ok: type_stats[qtype]["enr_correct"] += 1

        # Track changes
        if not orig_ok and enr_ok:
            improved_questions.append({
                "img_id": img_id, "question": question, "gt": gt,
                "orig": orig_ans, "enr": enr_ans, "type": qtype,
                "was_modified": was_modified,
            })
        elif orig_ok and not enr_ok:
            regressed_questions.append({
                "img_id": img_id, "question": question, "gt": gt,
                "orig": orig_ans, "enr": enr_ans, "type": qtype,
                "was_modified": was_modified,
            })

    per_image_results.append({
        "img_id": img_id, "orig_acc": img_orig_c / max(img_total, 1),
        "enr_acc": img_enr_c / max(img_total, 1), "n_questions": img_total,
        "was_modified": was_modified,
    })

    if (idx + 1) % 20 == 0:
        print(f"  Evaluated {idx+1}/{len(images)} images ({total_q} questions)...")
    time.sleep(0.2)


# ── Results ──
print(f"\n{'='*70}")
print(f"R-POPE (LLM-Judge) RESULTS — {total_q} questions across {len(per_image_results)} images")
print(f"{'='*70}")

if total_q > 0:
    orig_acc = orig_correct / total_q
    enr_acc = enr_correct / total_q
    delta = enr_correct - orig_correct

    print(f"\n  Original (BLIP-2) accuracy:  {orig_correct}/{total_q} ({orig_acc:.1%})")
    print(f"  Enriched (RelCheck) accuracy: {enr_correct}/{total_q} ({enr_acc:.1%})")
    print(f"  Delta: {'+' if delta >= 0 else ''}{delta} questions ({delta/total_q:+.1%})")

    if delta > 0:
        print(f"\n  >>> RelCheck IMPROVES accuracy by {delta/total_q:+.1%} <<<")
    elif delta == 0:
        print(f"\n  --- No change ---")
    else:
        print(f"\n  !!! RelCheck DECREASED accuracy by {delta/total_q:.1%} !!!")

    # ── Breakdown by question type ──
    print(f"\n  {'Type':<12} {'Original':<15} {'Enriched':<15} {'Delta':<10}")
    print(f"  {'─'*50}")
    for qtype in ["SPATIAL", "ACTION", "ATTRIBUTE", "OTHER"]:
        s = type_stats[qtype]
        if s["total"] == 0: continue
        oa = s["orig_correct"] / s["total"]
        ea = s["enr_correct"] / s["total"]
        d = s["enr_correct"] - s["orig_correct"]
        print(f"  {qtype:<12} {s['orig_correct']}/{s['total']} ({oa:.0%}){'':>3} "
              f"{s['enr_correct']}/{s['total']} ({ea:.0%}){'':>3} {'+' if d >= 0 else ''}{d}")

    # ── Improved questions ──
    if improved_questions:
        print(f"\n  IMPROVED ({len(improved_questions)} questions):")
        for q in improved_questions[:10]:
            mod = " [MODIFIED]" if q["was_modified"] else " [unchanged]"
            print(f"    [{q['type']}] {q['img_id'][:12]}{mod}")
            print(f"      Q: {q['question'][:70]}")
            print(f"      GT={q['gt']} | Orig={q['orig']} | Enr={q['enr']}")

    # ── Regressed questions ──
    if regressed_questions:
        print(f"\n  REGRESSED ({len(regressed_questions)} questions):")
        for q in regressed_questions[:10]:
            mod = " [MODIFIED]" if q["was_modified"] else " [unchanged]"
            print(f"    [{q['type']}] {q['img_id'][:12]}{mod}")
            print(f"      Q: {q['question'][:70]}")
            print(f"      GT={q['gt']} | Orig={q['orig']} | Enr={q['enr']}")

    # ── Net impact ──
    print(f"\n  Net: +{len(improved_questions)} improved, -{len(regressed_questions)} regressed")

    # ── Modified vs unmodified breakdown ──
    mod_imgs = [r for r in per_image_results if r["was_modified"]]
    unmod_imgs = [r for r in per_image_results if not r["was_modified"]]
    if mod_imgs:
        mod_orig = sum(r["orig_acc"] * r["n_questions"] for r in mod_imgs) / sum(r["n_questions"] for r in mod_imgs)
        mod_enr = sum(r["enr_acc"] * r["n_questions"] for r in mod_imgs) / sum(r["n_questions"] for r in mod_imgs)
        print(f"\n  Modified images ({len(mod_imgs)}):   {mod_orig:.1%} → {mod_enr:.1%}")
    if unmod_imgs:
        unmod_acc = sum(r["orig_acc"] * r["n_questions"] for r in unmod_imgs) / sum(r["n_questions"] for r in unmod_imgs)
        print(f"  Unmodified images ({len(unmod_imgs)}): {unmod_acc:.1%} (unchanged)")

print(f"\n{'='*70}")

# ── Save ──
save_path = "/content/drive/MyDrive/RelCheck_Data/enriched_100img_results.json"
save_data = {}
for img_id in images:
    r = final_results[img_id]
    save_data[img_id] = {
        "caption": r["caption"], "corrected": r["corrected"],
        "errors": r["errors"], "missing": r["missing"],
        "edit_rate": r["edit_rate"], "status": r["status"],
    }
with open(save_path, "w") as f:
    json.dump(save_data, f, indent=2, default=str)
print(f"Results saved to {save_path}")
